In [ ]:
# Import dependencies
import os
import time
import fitz  # PyMuPDF
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from transformers import AutoImageProcessor, AutoModel, CLIPProcessor, CLIPModel
import faiss
import timm

In [ ]:
def extract_pdf_images(pdf_path, output_dir="pdf_images"):
    """Extract all images from a PDF and save them to the specified directory"""
    os.makedirs(output_dir, exist_ok=True)
    doc = fitz.open(pdf_path)
    image_paths = []
    
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        img_list = page.get_images(full=True)
        
        for img_index, img in enumerate(img_list):
            xref = img[0]
            base_image = doc.extract_image(xref)
            img_data = base_image["image"]
            img_path = f"{output_dir}/page_{page_num}_img_{img_index}.png"
            
            with open(img_path, "wb") as f:
                f.write(img_data)
            image_paths.append(img_path)
    
    print(f"Extracted {len(image_paths)} images from {pdf_path} to {output_dir}")
    return image_paths


# Example: Extract images from a specified PDF
pdf_path = "long.pdf"  # Replace with your PDF path
image_paths = extract_pdf_images(pdf_path)

In [ ]:
# Import required libraries
import torch
from torchvision import models, transforms
from PIL import Image
import requests

# 1. Load pre-trained VGG16 model
model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
model.eval()  # Set to evaluation mode

# 2. Enhanced image preprocessing pipeline (to handle channel issues)
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.Lambda(lambda x: x.convert('RGB') if x.mode != 'RGB' else x),  # Force RGB conversion
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 3. Load local image (with exception handling)
img_path = r'long.png'
try:
    img = Image.open(img_path)
    print(f"Original image mode: {img.mode}, size: {img.size}")  # Debug info

    # Apply preprocessing
    input_tensor = preprocess(img)
    print(f"Preprocessed tensor shape: {input_tensor.shape}")  # Expected: [3, 224, 224]

except Exception as e:
    print(f"Image loading failed: {str(e)}")
    exit()

# Add batch dimension and check shape
input_batch = input_tensor.unsqueeze(0)  # Shape becomes [1, 3, 224, 224]
print(f"Input batch shape: {input_batch.shape}")  # Verify correctness

# 4. GPU acceleration setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_batch = input_batch.to(device)
model = model.to(device)
print(f"Using device: {device}")

# 5. Perform inference
with torch.no_grad():
    output = model(input_batch)

# 6. Decode prediction results (with offline fallback for labels)
try:
    # Download ImageNet class labels
    classes_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
    classes = requests.get(classes_url).text.splitlines()
except:
    # Offline fallback
    classes = [str(i) for i in range(1000)]  # Use index as labels

# Get and print results
probabilities = torch.nn.functional.softmax(output[0], dim=0)
top5_prob, top5_catid = torch.topk(probabilities, 5)

print("\nPrediction Results:")
for i in range(top5_prob.size(0)):
    print(f"Top-{i + 1}: {classes[top5_catid[i]]:<25} Confidence: {top5_prob[i].item() * 100:.2f}%")

In [ ]:
class FeatureExtractor:
    def _init_model(self):
        if self.method == "VGG16":
            # 1. Create model without classification head
            self.model = timm.create_model(
                'vgg16',
                pretrained=False,
                num_classes=0,  # Remove classification head
            )
            
            # 2. Load local weights and ignore mismatched keys
            state_dict = torch.load("local_models/vgg16/pytorch_model.bin")
            
            # 3. Filter out classification head parameters
            state_dict = {
                k: v for k, v in state_dict.items()
                if not k.startswith("head.fc")
            }
            
            # 4. Load with non-strict mode
            self.model.load_state_dict(state_dict, strict=False)
            
            self.model.eval()
            
            self.transform = transforms.Compose([
                transforms.Resize(224),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                )
            ])

In [ ]:
# Test whether local model loading is successful 

# Define local model paths
LOCAL_MODEL_PATHS = {
    "VGG16": "./vgg16/pytorch_model.bin",       # Local weights file for VGG16
    "DINOv2": "./dinov2-small",                  # Local model directory for DINOv2
    "CLIP": "./clip-vit-base-patch32"           # Local model directory for CLIP
}

# Test whether local models load successfully
def test_local_loading():
    try:
        # Test VGG16
        vgg_extractor = FeatureExtractor("VGG16", LOCAL_MODEL_PATHS["VGG16"])
        print("VGG16 local loading successful!")
        
        # Test DINOv2
        dinov2_extractor = FeatureExtractor("DINOv2", LOCAL_MODEL_PATHS["DINOv2"])
        print("DINOv2 local loading successful!")
        
        # Test CLIP
        clip_extractor = FeatureExtractor("CLIP", LOCAL_MODEL_PATHS["CLIP"])
        print("CLIP local loading successful!")
        
    except Exception as e:
        print(f"Loading failed: {str(e)}")


test_local_loading()

In [ ]:
import os
import torch
import timm
from transformers import AutoImageProcessor, AutoModel, CLIPProcessor, CLIPModel
from torchvision import transforms
from PIL import Image
import cv2
import numpy as np


class FeatureExtractor:
    """Supports five methods: SIFT / ORB / VGG16 / DINOv2 / CLIP"""

    def __init__(self, method, local_model_path=None):
        self.method = method
        self.local_model_path = local_model_path
        self.model = None
        self.processor = None
        self.transform = None
        self._init_model()

    def _init_model(self):
        if self.method == "VGG16":
            self.model = timm.create_model('vgg16', pretrained=False, num_classes=0).eval()

            if self.local_model_path and os.path.exists(self.local_model_path):
                state_dict = torch.load(self.local_model_path)
                self.model.load_state_dict(state_dict, strict=False)
            else:
                raise FileNotFoundError(
                    f"VGG16 local model path not found: {self.local_model_path}"
                )

            self.transform = transforms.Compose([
                transforms.Resize(224),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                )
            ])

        elif self.method == "DINOv2":
            if self.local_model_path and os.path.exists(self.local_model_path):
                self.processor = AutoImageProcessor.from_pretrained(self.local_model_path)
                self.model = AutoModel.from_pretrained(self.local_model_path).eval()
            else:
                raise FileNotFoundError(
                    f"DINOv2 local model path not found: {self.local_model_path}"
                )

        elif self.method == "CLIP":
            if self.local_model_path and os.path.exists(self.local_model_path):
                self.processor = CLIPProcessor.from_pretrained(self.local_model_path)
                self.model = CLIPModel.from_pretrained(self.local_model_path).eval()
            else:
                raise FileNotFoundError(
                    f"CLIP local model path not found: {self.local_model_path}"
                )

        elif self.method in ["SIFT", "ORB"]:
            pass

        else:
            raise ValueError(f"Unsupported method: {self.method}")

    def extract(self, img_path):
        img = Image.open(img_path).convert("RGB")

        if self.method == "VGG16":
            tensor = self.transform(img).unsqueeze(0)
            with torch.no_grad():
                return self.model(tensor).flatten().numpy()

        elif self.method == "DINOv2":
            inputs = self.processor(images=img, return_tensors="pt", do_resize=False)
            with torch.no_grad():
                return self.model(**inputs).last_hidden_state.mean(dim=1).numpy().flatten()

        elif self.method == "CLIP":
            inputs = self.processor(images=img, return_tensors="pt")
            with torch.no_grad():
                return self.model.get_image_features(**inputs).numpy().flatten()

        elif self.method == "SIFT":
            img_cv = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            sift = cv2.SIFT_create()
            kp, des = sift.detectAndCompute(img_cv, None)
            return des

        elif self.method == "ORB":
            img_cv = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            orb = cv2.ORB_create()
            kp, des = orb.detectAndCompute(img_cv, None)
            return des

        else:
            raise ValueError(f"Unsupported method: {self.method}")


# Define local model paths
LOCAL_MODEL_PATHS = {
    "VGG16": "./vgg16/pytorch_model.bin",
    "DINOv2": "./dinov2-small",
    "CLIP": "./clip-vit-base-patch32"
}


# Test whether local model loading is successful
def test_local_loading():
    try:
        # Test VGG16
        vgg_extractor = FeatureExtractor("VGG16", LOCAL_MODEL_PATHS["VGG16"])
        print("VGG16 local loading successful!")

        # Test DINOv2
        dinov2_extractor = FeatureExtractor("DINOv2", LOCAL_MODEL_PATHS["DINOv2"])
        print("DINOv2 local loading successful!")

        # Test CLIP
        clip_extractor = FeatureExtractor("CLIP", LOCAL_MODEL_PATHS["CLIP"])
        print("CLIP local loading successful!")

    except Exception as e:
        print(f"Loading failed: {str(e)}")


test_local_loading()

In [ ]:
import torch
import timm
from transformers import AutoImageProcessor, AutoModel, CLIPProcessor, CLIPModel
from torchvision import transforms
from PIL import Image
import cv2
import numpy as np


def sift_match(img1_path, img2_path, min_matches=15):
    """SIFT feature matching"""
    img1 = cv2.imread(img1_path, cv2.IMREAD_GRAYSCALE)
    img2 = cv2.imread(img2_path, cv2.IMREAD_GRAYSCALE)

    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(img1, None)
    kp2, des2 = sift.detectAndCompute(img2, None)

    if des1 is None or des2 is None:
        return False

    flann = cv2.FlannBasedMatcher(
        dict(algorithm=1, trees=5),
        dict(checks=50)
    )

    matches = flann.knnMatch(des1, des2, k=2)
    good = [m for m, n in matches if m.distance < 0.7 * n.distance]

    return len(good) >= min_matches


def orb_match(img1_path, img2_path, min_matches=30):
    """ORB feature matching"""
    img1 = cv2.imread(img1_path, cv2.IMREAD_GRAYSCALE)
    img2 = cv2.imread(img2_path, cv2.IMREAD_GRAYSCALE)

    orb = cv2.ORB_create()
    kp1, des1 = orb.detectAndCompute(img1, None)
    kp2, des2 = orb.detectAndCompute(img2, None)

    if des1 is None or des2 is None:
        return False

    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des1, des2)

    return len(matches) >= min_matches

In [ ]:
class OracleRetrievalSystem:
    def __init__(self, method, local_model_path=None):
        self.method = method
        self.extractor = FeatureExtractor(method, local_model_path) if method in ["VGG16", "DINOv2", "CLIP"] else None
        self.index = None
        self.image_paths = []

    def build_index(self, image_paths):
        """Build FAISS index for deep learning-based methods"""
        self.image_paths = image_paths

        if self.method in ["VGG16", "DINOv2", "CLIP"]:
            features = [self.extractor.extract(path) for path in image_paths]
            features = np.array(features)

            dim = features[0].shape[0]

            self.index = faiss.IndexFlatL2(dim) if self.method == "DINOv2" else faiss.IndexFlatIP(dim)

            if self.method != "DINOv2":
                faiss.normalize_L2(features)

            self.index.add(features)


# Define local model paths
LOCAL_MODEL_PATHS = {
    "VGG16": "./vgg16/pytorch_model.bin",
    "DINOv2": "./dinov2-small",
    "CLIP": "./clip-vit-base-patch32"
}


# Initialize system
algorithms = ["SIFT", "ORB", "VGG16", "DINOv2", "CLIP"]
systems = {
    name: OracleRetrievalSystem(name, LOCAL_MODEL_PATHS.get(name))
    for name in algorithms
}

# Build index for deep learning-based methods (assuming image_paths is defined)
for name in ["VGG16", "DINOv2", "CLIP"]:
    systems[name].build_index(image_paths)

In [ ]:
class OracleRetrievalSystem:
    def __init__(self, method, local_model_path=None):
        self.method = method
        self.extractor = FeatureExtractor(method, local_model_path) if method in ["VGG16", "DINOv2", "CLIP"] else None
        self.index = None
        self.image_paths = []

    def build_index(self, image_paths):
        """Build FAISS index for deep learning-based methods"""
        self.image_paths = image_paths

        if self.method in ["VGG16", "DINOv2", "CLIP"]:
            features = [self.extractor.extract(path) for path in image_paths]
            features = np.array(features)

            dim = features[0].shape[0]

            self.index = faiss.IndexFlatL2(dim) if self.method == "DINOv2" else faiss.IndexFlatIP(dim)

            if self.method != "DINOv2":
                faiss.normalize_L2(features)

            self.index.add(features)

    def query(self, query_path, top_k=5):
        """Perform retrieval query"""
        if self.method in ["SIFT", "ORB"]:
            matches = []
            for path in self.image_paths:
                if self.method == "SIFT":
                    match = sift_match(query_path, path)
                else:
                    match = orb_match(query_path, path)
                matches.append(match)

            sorted_indices = np.argsort(-np.array(matches))[:top_k]
            return [self.image_paths[i] for i in sorted_indices if matches[i]]

        else:
            query_feat = self.extractor.extract(query_path)
            query_feat = np.expand_dims(query_feat, 0)

            if self.method != "DINOv2":
                faiss.normalize_L2(query_feat)

            distances, indices = self.index.search(query_feat, top_k)
            return [self.image_paths[i] for i in indices[0]]

    def evaluate(self, test_pairs, top_k=5):
        """Evaluate retrieval performance"""
        y_true, y_pred = [], []
        total_time = 0.0

        for query_path, target_paths in test_pairs:
            start = time.time()
            retrieved = self.query(query_path, top_k)
            total_time += time.time() - start

            # Determine whether at least one true target is retrieved
            hit = any(path in retrieved for path in target_paths)

            y_true.append(1 if hit else 0)
            y_pred.append(1 if hit else 0)

        return {
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred),
            "Recall": recall_score(y_true, y_pred),
            "F1": f1_score(y_true, y_pred),
            "Avg Time (ms)": (total_time / len(test_pairs)) * 1000
        }

In [ ]:
# Generate test pairs (assuming the positive sample for each query image is the next image)
test_pairs = []
for i in range(len(image_paths)):
    query = image_paths[i]
    target = image_paths[(i + 1) % len(image_paths)]  # simple circular pairing
    test_pairs.append((query, [target]))

In [ ]:
algorithms = ["SIFT", "ORB", "VGG16", "DINOv2", "CLIP"]
systems = {name: OracleRetrievalSystem(name, LOCAL_MODEL_PATHS.get(name)) for name in algorithms}

for name in ["VGG16", "DINOv2", "CLIP"]:
    systems[name].build_index(image_paths)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Font file path (based on fc-list result)
font_path = "/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc"

# Load font manually
prop = fm.FontProperties(fname=font_path)

# Set global Matplotlib font
plt.rcParams['font.family'] = prop.get_name()

# Plot test
plt.figure()
plt.text(
    0.5, 0.5,
    "Font Test: WenQuanYi Zen Hei",
    fontproperties=prop,
    ha='center',
    va='center'
)
plt.show()

In [ ]:
# 4.4 Visualization of comparison results

# Configure Matplotlib font
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei']
plt.rcParams['axes.unicode_minus'] = False

# Generate comparison table
import pandas as pd

df = pd.DataFrame(results).T
df = df[["Accuracy", "Precision", "Recall", "F1", "Avg Time (ms)"]]

print("\n=== Algorithm Performance Comparison ===")
print(df)

# Plot performance metrics comparison
plt.figure(figsize=(12, 6))
df.drop("Avg Time (ms)", axis=1).plot(kind='bar', rot=0)

plt.title("Oracle Bone Image Retrieval Algorithm Performance Comparison (Top-5)")
plt.ylabel("Score")
plt.ylim(0, 1.1)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# Plot average retrieval time comparison
plt.figure(figsize=(8, 5))
df["Avg Time (ms)"].plot(kind='bar', color='purple', rot=0)

plt.title("Average Retrieval Time Comparison")
plt.ylabel("Time (ms)")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# 4.3 Perform evaluation
results = {}
for name in algorithms:
    print(f"Evaluating algorithm: {name}")
    results[name] = systems[name].evaluate(test_pairs)

In [ ]:
def interactive_search():
    """User inputs any image path for retrieval"""
    # Input query image path
    query_path = input("./long.png): ").strip()

    # Check path validity
    if not os.path.exists(query_path):
        print("Error: File does not exist!")
        return

    # Display query image
    try:
        query_img = Image.open(query_path)
        plt.figure(figsize=(4, 4))
        plt.imshow(query_img)
        plt.title("Query Image: " + os.path.basename(query_path))
        plt.axis('off')
        plt.show()
    except Exception as e:
        print(f"Image loading failed: {e}")
        return

    # Perform retrieval
    print("\n=== Retrieval Results ===")
    for name in algorithms:
        start = time.time()
        results = systems[name].query(query_path)
        elapsed = (time.time() - start) * 1000

        print(f"\nAlgorithm: {name} (Time: {elapsed:.1f} ms)")

        if len(results) == 0:
            print("No matching results found")
            continue

        # Display Top-5 results
        plt.figure(figsize=(15, 3))
        for i, path in enumerate(results[:5]):
            img = Image.open(path)
            plt.subplot(1, 5, i + 1)
            plt.imshow(img)
            plt.title(f"Top {i + 1}\n{os.path.basename(path)}")
            plt.axis('off')

        plt.tight_layout()
        plt.show()


# --------------------- Run Example ---------------------
interactive_search()

In [ ]:
def interactive_search(query_image_path):
    """User inputs a query image and returns results from all algorithms"""
    query_img = Image.open(query_image_path)
    plt.imshow(query_img)
    plt.title("Query Image")
    plt.axis('off')
    plt.show()

    print("=== Retrieval Results ===")
    for name in algorithms:
        start = time.time()
        results = systems[name].query(query_image_path)
        elapsed = (time.time() - start) * 1000

        print(f"\nAlgorithm: {name} (Time: {elapsed:.1f} ms)")

        plt.figure(figsize=(15, 3))
        for i, path in enumerate(results[:5]):
            img = Image.open(path)
            plt.subplot(1, 5, i + 1)
            plt.imshow(img)
            plt.title(f"Top {i + 1}")
            plt.axis('off')
        plt.show()


# Example: query the first image
# interactive_search(image_paths[0])

# Safe execution example with file check
query_path = 'long.png'
if os.path.exists(query_path):
    interactive_search(query_path)
else:
    print(f"Error: file {query_path} does not exist")